# Python Data Pipeline Engineering - Lab
ETL Pipeline แบบ Incremental และ Idempotent

In [ ]:
from pipeline import PipelineConfig, run_pipeline

config = PipelineConfig(
    input_path="Python_Data_Pipeline_Lab_Data.xlsx",
    output_database="retail_dw.db",
    batch_list=(1, 2, 3),
    error_mode="quarantine",
)

## รอบการรันตามโจทย์
1. batch_1
2. batch_1 ซ้ำ
3. batch_2
4. batch_3

In [ ]:
results = []
for b in [1, 1, 2, 3]:
    results.append(run_pipeline(config, b))

import pandas as pd
pd.DataFrame(results)

## ตรวจสอบผลลัพธ์ใน SQLite

In [ ]:
import sqlite3
con = sqlite3.connect("retail_dw.db")

for table in ["dim_customer", "dim_product", "dim_date", "fact_sales"]:
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(table, n)

print("Fact duplicates:", con.execute("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM fact_sales").fetchone()[0])
print("Net sales:", con.execute("SELECT ROUND(SUM(net_amount), 2) FROM fact_sales").fetchone()[0])
con.close()

## ดูข้อมูลที่ถูก Quarantine

In [ ]:
q = pd.read_csv("quarantine.csv")
q["reason_code"].value_counts()

## Acceptance Test
- order_id ไม่ซ้ำ
- Foreign key ครบ
- quantity/unit_price/net_amount ไม่ติดลบ
- batch_1 ซ้ำไม่เพิ่ม Fact
- reject ทุกแถวมี reason_code